# 03 — Local prediction tests

Score saved `model.joblib` the same way you will later call a Vertex endpoint.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd

from trainer.data import FEATURE_COLUMNS, load_raw_csv, clean_churn_frame, split_features_and_target
from trainer.predict import predict_instances

MODEL_PATH = Path("../artifacts/model/model.joblib")
model = joblib.load(MODEL_PATH)
FEATURE_COLUMNS

In [ ]:
raw = load_raw_csv(Path("../data/raw/Telco-Customer-Churn.csv"))
clean = clean_churn_frame(raw)
X, y = split_features_and_target(clean)

high_risk = X.loc[(X["Contract"] == "Month-to-month") & (X["tenure"] <= 2)].head(3)
low_risk = X.loc[(X["Contract"] == "Two year") & (X["tenure"] >= 60)].head(3)

print("high risk")
display(pd.DataFrame(predict_instances(model, high_risk.to_dict(orient="records"))))
print("low risk")
display(pd.DataFrame(predict_instances(model, low_risk.to_dict(orient="records"))))

## Contract checks

Missing columns must fail. Extra identifier columns are ignored only if you go through `predict_instances` / `FEATURE_COLUMNS`.

In [ ]:
try:
    predict_instances(model, [{"tenure": 1}])
except ValueError as exc:
    print("expected error:", exc)

In [ ]:
payload = high_risk.iloc[0].to_dict()
payload_path = Path("../artifacts/sample_instance.json")
import json
payload_path.write_text(json.dumps(payload, indent=2) + "\n")
print("wrote", payload_path)